# Lab 08 Challenge: worked solution

For the instructor. Each task uses only what Lab 08 showed. Run the setup and config cells, then the cell that
rebuilds the lab's pieces (the same code as the notebook, collected in one place). The notes below come from this
notebook's saved run (`gpt-4o-mini`, `text-embedding-3-small`).

In [1]:
# Setup: on Colab, copy the repository and install the packages. Locally (after uv sync) nothing is installed.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/agentic_ai_conversational_agents").exists():
        !git clone -q --depth 1 https://github.com/tatwan/agentic_ai_conversational_agents.git /content/agentic_ai_conversational_agents
    %cd /content/agentic_ai_conversational_agents
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: agentic_ai_conversational_agents


In [2]:
# Config: model names come from models.env; the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name in ("CHAT_MODEL", "REASONING_EFFORT", "EMBED_MODEL"):
    print(f"{name:18}{settings[name] or '(not set)'}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

CHAT_MODEL        gpt-4o-mini
REASONING_EFFORT  (not set)
EMBED_MODEL       text-embedding-3-small
OPENAI_API_KEY    found in .env


The pieces from the lab: the store, `search`, the tool, the copilot with the abstention line, and the checks.

In [3]:
import re
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from shared.settings import chat_model

handbook = (ROOT / "data" / "handbook" / "support_handbook.md").read_text(encoding="utf-8")
splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("##", "section")], strip_headers=False)
documents = []
for part in splitter.split_text(handbook):
    if "section" in part.metadata:
        source = part.page_content.split("Source: ")[1].split()[0]
        text = part.page_content.replace(f"Source: {source}", "").strip()
        documents.append(Document(page_content=text, metadata={"source": source, "section": part.metadata["section"]}))
vector_store = InMemoryVectorStore(OpenAIEmbeddings(model=settings["EMBED_MODEL"]))
vector_store.add_documents(documents)

def search(query, k=4):
    return vector_store.similarity_search_with_score(query, k=k)

@tool(parse_docstring=True, response_format="content_and_artifact")
def search_policies(query: str) -> tuple[str, list[dict]]:
    """Search the support handbook for the four passages closest to a query.

    Call it before answering any question about shop rules.
    Each passage starts with its source file in square brackets.

    Args:
        query: What the rep needs to know, in a short sentence.
    """
    hits = search(query)
    content = "\n\n".join(f"[{doc.metadata['source']}]\n{doc.page_content}" for doc, score in hits)
    return content, [{"source": doc.metadata["source"], "score": round(score, 4)} for doc, score in hits]

ABSTAIN_PROMPT = """You are the Chinook support copilot. You help support reps answer customer emails.
Use search_policies for any question about the shop's rules, and answer only from the passages it returns.
After each rule you use, name its file in square brackets, for example [policy_refunds.md].
If the passages do not answer the question, say that the handbook does not cover it and suggest asking the support manager. Do not guess."""
model = chat_model(settings)
copilot = create_agent(model, tools=[search_policies], system_prompt=ABSTAIN_PROMPT)

CITATION = re.compile(r"\[(policy_[a-z_]+\.md)\]")

def cited_files(text):
    return sorted(set(CITATION.findall(text)))

def retrieved_files(messages):
    return [hit["source"] for message in messages if message.type == "tool" for hit in message.artifact]

def checks(messages):
    reply = messages[-1].text
    return {
        "searched": any(message.type == "tool" for message in messages),
        "cited": cited_files(reply),
        "cited files retrieved": set(cited_files(reply)) <= set(retrieved_files(messages)),
        "abstained": "support manager" in reply.lower() and not cited_files(reply),
    }

QUESTIONS = {
    "duplicate charge": "A customer was charged twice for the same album on the same day. What do we do?",
    "price match": "A customer found the same album cheaper at another store. Do we match a lower price?",
}
print(len(documents), "documents in the store")

20 documents in the store


## 1. A score threshold instead of the abstention line

First the top score for each question, and for the short query from Part B.

In [4]:
SHORT_QUERY = "charged twice for the same album"
for label, q in [*QUESTIONS.items(), ("short query", SHORT_QUERY)]:
    doc, score = search(q, k=1)[0]
    print(f"{label:16} top {score:.4f}  {doc.metadata['source']}")

duplicate charge top 0.4807  policy_refunds.md


price match      top 0.4118  policy_exchanges.md


short query      top 0.3873  policy_damaged_media.md


The duplicate charge scores 0.4807 and the price question 0.4118, so a threshold of 0.45 sits between them. The short query's top score is 0.3873, from `policy_damaged_media.md`.

In [5]:
ABSTAIN = "The handbook does not cover this."
TWO_STEP_PROMPT = ("You are the Chinook support copilot. You help support reps answer customer emails. "
                   "Answer only from the handbook passages in the message.")

def answer_or_abstain(query, threshold):
    hits = search(query)
    if hits[0][1] < threshold:
        return f"(top {hits[0][1]:.4f}) {ABSTAIN}"
    passages = "\n\n".join(f"[{doc.metadata['source']}]\n{doc.page_content}" for doc, score in hits)
    reply = model.invoke([("system", TWO_STEP_PROMPT), ("user", f"Handbook passages:\n{passages}\n\nRep: {query}")])
    return f"(top {hits[0][1]:.4f}) {reply.text}"

for label, q in [*QUESTIONS.items(), ("short query", SHORT_QUERY)]:
    print(f"{label}: {answer_or_abstain(q, 0.45)}\n")

duplicate charge: (top 0.4807) You should refund the later invoice in full. Make sure to attach both invoice numbers.



price match: (top 0.4118) The handbook does not cover this.



short query: (top 0.3873) The handbook does not cover this.



The threshold answers the duplicate charge and abstains on the price question, as intended. It also abstains on the short query, which the handbook answers: the refunds section is there, fourth at 0.3391, but the top score is below 0.45. A cosine score depends on the words of the query and is not a probability, so a threshold tuned on one wording fails on another. Abstention belongs to the model's reading of the passages, checked in code (Parts E and F).

## 2. Optional: search some sections only

`filter=` takes a function that receives each `Document` and returns `True` to keep it. The filter runs inside the
search, so only the kept sections can come back.

In [6]:
def search(query, k=4, sections=None):
    keep = None
    if sections is not None:
        def keep(doc):
            return doc.metadata["section"] in sections
    return vector_store.similarity_search_with_score(query, k=k, filter=keep)

for doc, score in search(QUESTIONS["price match"], sections={"Payment", "Exchanges"}):
    print(f"{score:.4f}  {doc.metadata['source']:22} {doc.metadata['section']}")

0.4118  policy_exchanges.md    Exchanges
0.2785  policy_payment.md      Payment


Two results, however large `k` is: exchanges (0.4118) and payment (0.2785). Neither answers the price question. A filter decides which documents can come back, not whether any of them answers; the copilot still has to abstain.

## 3. Optional: the copilot on new questions

Written before running: two questions the handbook answers, two it does not, and what each should give.

In [7]:
NEW_EXPECTED = {
    "Can a customer pay with a gift card and a card on the same order?":
        {"searched": True, "cited": ["policy_payment.md"], "cited files retrieved": True, "abstained": False},
    "How long does a customer have to download a digital album?":
        {"searched": True, "cited": ["policy_digital_downloads.md"], "cited files retrieved": True, "abstained": False},
    "Can a customer pay for an album in monthly instalments?":
        {"searched": True, "cited": [], "cited files retrieved": True, "abstained": True},
    "Do we offer gift wrapping for vinyl orders?":
        {"searched": True, "cited": [], "cited files retrieved": True, "abstained": True},
}
copilot = create_agent(model, tools=[search_policies], system_prompt=ABSTAIN_PROMPT)
for q, expected in NEW_EXPECTED.items():
    passed = 0
    for _ in range(3):
        messages = copilot.invoke({"messages": [("user", q)]})["messages"]
        result = checks(messages)
        passed += result == expected
        if result != expected:
            print("   failed:", result, "|", messages[-1].text[:150])
    print(f"{passed} of 3 | {q}")

3 of 3 | Can a customer pay with a gift card and a card on the same order?


3 of 3 | How long does a customer have to download a digital album?


3 of 3 | Can a customer pay for an album in monthly instalments?


3 of 3 | Do we offer gift wrapping for vinyl orders?


All four questions pass 3 of 3: the two covered questions cite `policy_payment.md` and `policy_digital_downloads.md`, the two uncovered ones abstain with no citation. Written before the run, these cases test the abstention line on questions it was not written from.